# SC-VAE — آموزش سبک روی Google Colab

اجرای سبکِ آموزش (Stage 1) مدل SC-VAE روی Colab با GPU:
کلون مخزن ← نصب وابستگی‌ها ← آماده‌سازی دادهٔ کوچک ← آموزش با تنظیماتِ مناسبِ همگرایی ← مشاهدهٔ نتایج.

> ⚙️ قبل از اجرا: از منوی **Runtime → Change runtime type** گزینهٔ **GPU** (مثلاً T4) را انتخاب کن.


## ۱) دریافت کد و نصب وابستگی‌ها

In [ ]:
!git clone https://github.com/e-shams-d/SC-VAE.git

In [ ]:
import os
os.chdir('/content/SC-VAE')
!pip install -q omegaconf easydict tensorboardX

## ۲) آماده‌سازی دادهٔ کوچک

برای دیدن همگرایی با هزینهٔ کم، یک مجموعهٔ کوچکِ **ثابت** از تصاویر واقعی ۲۵۶×۲۵۶ دانلود می‌کنیم و
فایل‌های لیستِ مخزن را بازنویسی می‌کنیم تا فقط به همین تصاویر اشاره کنند.

> برای **آموزش واقعی**، این سلول را با تصاویر واقعی FFHQ جایگزین کن
> (تصاویر را در `dataset/FFHQ/resized/` بگذار و نام‌هایشان را در فایل‌های لیست بنویس).


In [ ]:
import os, requests

root = 'dataset/FFHQ/resized'
os.makedirs(root, exist_ok=True)

N = 64
ok = []
for i in range(N):
    name = f'{i:05d}.jpg'
    try:
        r = requests.get(f'https://picsum.photos/seed/{i}/256/256',
                         headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
        if r.status_code == 200 and len(r.content) > 1000:
            open(f'{root}/{name}', 'wb').write(r.content)
            ok.append(name)
    except Exception as e:
        print('skip', name, e)

# rewrite the repo list files so they reference ONLY the images we actually have
for fn in ['ffhqtrain.txt', 'ffhqvalidation.txt']:
    open(f'img_datasets/assets/{fn}', 'w').write('\n'.join(ok) + '\n')

print(f'downloaded {len(ok)}/{N} images; train/val list files rewritten')

## ۳) آموزش

تنظیماتِ یک اجرای سبکِ نمایشِ همگرایی (روی T4 حدود ۶–۸ دقیقه):

| پارامتر | مقدار | توضیح |
|---------|-------|-------|
| `experiment.epochs` | 30 | برای دیدن روند کاهش loss |
| `experiment.batch_size` | 4 | امن برای حافظهٔ T4 |
| `optimizer.init_lr` | 2.0e-4 | یادگیری محسوس (gradient clipping مانع انفجار است) |
| `optimizer.warmup.min_lr` | 1.0e-5 | کاهش تدریجی LR (cosine) |

👀 خط‌های `[epoch N] validation reconstruction loss:` باید **کاهش** پیدا کنند.
🆘 اگر `NaN` دیدی، فقط `optimizer.init_lr=2.0e-4` را به `1.0e-4` کم کن.


In [ ]:
!python main-stage1.py --dataset FFHQ \
  --model-config ./configs/ffhq/stage1/ffhq256-scvae16x16.yaml \
  --device cuda --num-workers 2 \
  experiment.epochs=30 experiment.batch_size=4 \
  optimizer.init_lr=2.0e-4 optimizer.warmup.min_lr=1.0e-5

## ۴) مشاهدهٔ نتایج (checkpointها + TensorBoard)

In [ ]:
# checkpointهای ذخیره‌شده (best.pt / last.pt)
!ls -lh results/models/FFHQ/*/

# نمودارهای loss + تصاویر بازسازی/دیکشنری
%load_ext tensorboard
%tensorboard --logdir results/logs